In [0]:
from pyspark.sql.functions import col, coalesce, lit, when, trim, upper, regexp_replace, expr
from pyspark.sql.types import StringType, IntegerType, DoubleType

def safe_timestamp(column_name):
    return expr(f"try_to_timestamp({column_name}, 'yyyy-MM-dd HH:mm:ss')")

# -----------------------------
# Products Silver Table
# -----------------------------
try:
    products_df = spark.table("ecommerce_catalog.bronze.products_table").select(
        col("product_id").cast(StringType()),
        regexp_replace(
            upper(coalesce(
                when(trim(col("product_category_name")) == "", None)
                .otherwise(trim(col("product_category_name"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("product_category_name"),
        coalesce(
            when(trim(col("product_weight_g")) == "", None)
            .otherwise(col("product_weight_g").cast(DoubleType()).cast(IntegerType())),
            lit(0)
        ).alias("product_weight_g"),
        coalesce(
            when(trim(col("product_length_cm")) == "", None)
            .otherwise(col("product_length_cm").cast(DoubleType()).cast(IntegerType())),
            lit(0)
        ).alias("product_length_cm"),
        coalesce(
            when(trim(col("product_height_cm")) == "", None)
            .otherwise(col("product_height_cm").cast(DoubleType()).cast(IntegerType())),
            lit(0)
        ).alias("product_height_cm"),
        coalesce(
            when(trim(col("product_width_cm")) == "", None)
            .otherwise(col("product_width_cm").cast(DoubleType()).cast(IntegerType())),
            lit(0)
        ).alias("product_width_cm")
    ).dropDuplicates(["product_id"])

    products_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.silver.products_clean")
    print("Products Silver Table created successfully!")

except Exception as e:
    print(f" Error creating Products Silver Table: {e}")


# -----------------------------
# Customers Silver Table
# -----------------------------
try:
    customers_df = spark.table("ecommerce_catalog.bronze.customers_table").select(
        col("customer_id").cast(StringType()),
        col("customer_unique_id").cast(StringType()),
        coalesce(
            when(trim(col("customer_zip_code_prefix")) == "", None)
            .otherwise(col("customer_zip_code_prefix").cast(IntegerType())),
            lit(0)
        ).alias("customer_zip_code_prefix"),
        regexp_replace(
            upper(coalesce(
                when(trim(col("customer_city")) == "", None)
                .otherwise(trim(col("customer_city"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("customer_city"),
        regexp_replace(
            upper(coalesce(
                when(trim(col("customer_state")) == "", None)
                .otherwise(trim(col("customer_state"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("customer_state")
    ).dropDuplicates(["customer_id"])

    customers_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.silver.customers_clean")
    print("Customers Silver Table created successfully!")

except Exception as e:
    print(f" Error creating Customers Silver Table: {e}")


# -----------------------------
# Sellers Silver Table
# -----------------------------
try:
    sellers_df = spark.table("ecommerce_catalog.bronze.sellers_table").select(
        col("seller_id").cast(StringType()),
        coalesce(
            when(trim(col("seller_zip_code_prefix")) == "", None)
            .otherwise(col("seller_zip_code_prefix").cast(IntegerType())),
            lit(0)
        ).alias("seller_zip_code_prefix"),
        regexp_replace(
            upper(coalesce(
                when(trim(col("seller_city")) == "", None)
                .otherwise(trim(col("seller_city"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("seller_city"),
        regexp_replace(
            upper(coalesce(
                when(trim(col("seller_state")) == "", None)
                .otherwise(trim(col("seller_state"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("seller_state")
    ).dropDuplicates(["seller_id"])

    sellers_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.silver.sellers_clean")
    print("Sellers Silver Table created successfully!")

except Exception as e:
    print(f" Error creating Sellers Silver Table: {e}")


# -----------------------------
# Orders Silver Table
# -----------------------------
try:
    orders_df = spark.table("ecommerce_catalog.bronze.orders_table").select(
        col("order_id").cast(StringType()),
        col("customer_id").cast(StringType()),
        coalesce(col("order_status"), lit("unknown")).alias("order_status"),
        safe_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
        safe_timestamp("order_approved_at").alias("order_approved_at"),
        safe_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        safe_timestamp("order_delivered_customer_date").alias("order_delivered_customer_date"),
        safe_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_date")
    ).dropDuplicates(["order_id"])

    orders_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.silver.orders_clean")
    print("Orders Silver Table created successfully!")

except Exception as e:
    print(f"Error creating Orders Silver Table: {e}")


# -----------------------------
# Order Payments Silver Table
# -----------------------------
try:
    payments_df = spark.table("ecommerce_catalog.bronze.order_payments_table").select(
        col("order_id").cast(StringType()),
        coalesce(
            when(trim(col("payment_sequential")) == "", None)
            .otherwise(col("payment_sequential").cast(DoubleType()).cast(IntegerType())),
            lit(1)
        ).alias("payment_sequential"),
        coalesce(col("payment_type"), lit("UNKNOWN")).alias("payment_type"),
        coalesce(
            when(trim(col("payment_installments")) == "", None)
            .otherwise(col("payment_installments").cast(DoubleType()).cast(IntegerType())),
            lit(1)
        ).alias("payment_installments"),
        coalesce(
            when(trim(col("payment_value")) == "", None)
            .otherwise(col("payment_value").cast(DoubleType())),
            lit(0.0)
        ).alias("payment_value")
    ).dropDuplicates(["order_id","payment_sequential"])

    payments_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.silver.order_payments_clean")
    print("Order Payments Silver Table created successfully!")

except Exception as e:
    print(f"Error creating Order Payments Silver Table: {e}")

# -----------------------------
# Order Items Silver Table
# -----------------------------
try:
    order_items_df = spark.table("ecommerce_catalog.bronze.order_items_table").select(
        col("order_id").cast(StringType()),
        col("order_item_id").cast(StringType()),
        col("product_id").cast(StringType()),
        col("seller_id").cast(StringType()),
        
        # shipping_limit_date: safely convert to timestamp
        expr("try_to_timestamp(shipping_limit_date, 'yyyy-MM-dd HH:mm:ss')")
        .alias("shipping_limit_date"),
        
        # price cleaning
        coalesce(
            when(trim(col("price")) == "", None)
            .otherwise(col("price").cast(DoubleType())),
            lit(0.0)
        ).alias("price"),
        
        # freight_value cleaning
        coalesce(
            when(trim(col("freight_value")) == "", None)
            .otherwise(col("freight_value").cast(DoubleType())),
            lit(0.0)
        ).alias("freight_value")
    ) \
    .dropDuplicates(["order_id", "order_item_id"])   # Remove duplicates

    # Write to Silver table
    order_items_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce_catalog.silver.order_items_clean")

    print("Order Items Silver Table created successfully!")

except Exception as e:
    print(f"Error creating Order Items Silver Table: {e}")


# -----------------------------
#  Product Category Name Translation Silver Table
# -----------------------------
try:
    product_category_translation_df = spark.table("ecommerce_catalog.bronze.product_category_translation_table").select(
        
        # Clean product_category_name
        regexp_replace(
            upper(coalesce(
                when(trim(col("product_category_name")) == "", None)
                .otherwise(trim(col("product_category_name"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("product_category_name"),
        
        # Clean product_category_name_english
        regexp_replace(
            upper(coalesce(
                when(trim(col("product_category_name_english")) == "", None)
                .otherwise(trim(col("product_category_name_english"))),
                lit("NONE")
            )),
            "[^a-zA-Z0-9 ]", ""
        ).alias("product_category_name_english")
        
    ) \
    .dropDuplicates(["product_category_name"])   # Remove duplicates

    # Write to Silver table
    product_category_translation_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce_catalog.silver.product_category_name_translation_clean")

    print("Product Category Name Translation Silver Table created successfully!")

except Exception as e:
    print(f"Error creating Product Category Name Translation Silver Table: {e}")

#-------------------------------------
#to check null values in all the tables
#----------------------------------------
tables = [
    "products_clean",
    "customers_clean",
    "sellers_clean",
    "orders_clean",
    "order_items_clean",
    "order_payments_clean",
    "product_category_name_translation_clean"
]

for table in tables:
    print(f"\nChecking empty spaces in table: {table}")
    
    df = spark.table(f"ecommerce_catalog.silver.{table}")
    
    for column, dtype in df.dtypes:
        
        # Check only string columns
        if dtype == "string":
            empty_count = df.filter(df[column] == "").count()
            
            if empty_count > 0:
                print(f"{column} -> Empty spaces: {empty_count}")

#------------------------------
#To check duplicates count
#-------------------------------
tables_keys = {
    "products_clean": ["product_id"],
    "customers_clean": ["customer_id"],
    "sellers_clean": ["seller_id"],
    "orders_clean": ["order_id"],
    "order_items_clean": ["order_id", "order_item_id"],
    "order_payments_clean": ["order_id", "payment_sequential"],
    "product_category_name_translation_clean": ["product_category_name"]
}
for table, keys in tables_keys.items():
    print(f"\nChecking duplicates in table: {table}")
    df = spark.table(f"ecommerce_catalog.silver.{table}")
    duplicates = (
        df.groupBy(keys)
        .count()
        .filter("count > 1")
    )
    duplicate_count = duplicates.count()
    if duplicate_count == 0:
        print("No duplicates found")
    else:
        print(f"{duplicate_count} duplicate records found")
        duplicates.show()

#-----------------------------------------
#just to disply the data
#-----------------------------------------

spark.table("ecommerce_catalog.silver.products_clean").show(10000, False)

#-----------------------------------
#Data Type Validation
#Expected schema for each Silver table
#-----------------------------------

expected_schemas = {

    "products_clean": {
        "product_id": "string",
        "product_category_name": "string",
        "product_weight_g": "int",
        "product_length_cm": "int",
        "product_height_cm": "int",
        "product_width_cm": "int"
    },

    "customers_clean": {
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "int",
        "customer_city": "string",
        "customer_state": "string"
    },

    "sellers_clean": {
        "seller_id": "string",
        "seller_zip_code_prefix": "int",
        "seller_city": "string",
        "seller_state": "string"
    },

    "orders_clean": {
        "order_id": "string",
        "customer_id": "string",
        "order_status": "string",
        "order_purchase_timestamp": "timestamp",
        "order_approved_at": "timestamp",
        "order_delivered_carrier_date": "timestamp",
        "order_delivered_customer_date": "timestamp",
        "order_estimated_delivery_date": "timestamp"
    },

    "order_items_clean": {
        "order_id": "string",
        "order_item_id": "string",
        "product_id": "string",
        "seller_id": "string",
        "shipping_limit_date": "timestamp",
        "price": "double",
        "freight_value": "double"
    },

    "order_payments_clean": {
        "order_id": "string",
        "payment_sequential": "int",
        "payment_type": "string",
        "payment_installments": "int",
        "payment_value": "double"
    },

    "product_category_name_translation_clean": {
        "product_category_name": "string",
        "product_category_name_english": "string"
    }
}
#----------------------------------------
# Validate datatypes for all the columns
#----------------------------------------

for table, expected_schema in expected_schemas.items():

    print("\n==============================")
    print(f"Validating Table: {table}")
    print("==============================")

    df = spark.table(f"ecommerce_catalog.silver.{table}")

    actual_schema = dict(df.dtypes)

    for column, expected_type in expected_schema.items():

        actual_type = actual_schema.get(column)

        if actual_type == expected_type:
            print(f" {column} -> Correct ({actual_type})")
        else:
            print(f" {column} -> Expected: {expected_type}, Found: {actual_type}")

#-------------------------------------
# Special character cleaning validation
#--------------------------------------


from pyspark.sql.functions import col

tables = [
    "products_clean",
    "customers_clean",
    "sellers_clean",
    "orders_clean",
    "order_items_clean",
    "order_payments_clean",
    "product_category_name_translation_clean"
]

for table in tables:

    print("\n==============================")
    print(f"Checking Special Characters in: {table}")
    print("==============================")

    df = spark.table(f"ecommerce_catalog.silver.{table}")

    for column, dtype in df.dtypes:

        if dtype == "string":

            special_count = df.filter(
                col(column).rlike("[^a-zA-Z0-9_ ]")
            ).count()

            if special_count > 0:
                print(f" {column} -> {special_count} rows contain special characters")
            else:
                print(f" {column} -> No special characters")

#--------------------
#Timestamp validation
#--------------------

from pyspark.sql.functions import col, coalesce

orders_df = spark.table("ecommerce_catalog.silver.orders_clean")

orders_fixed = orders_df.withColumn(
    "order_approved_at",
    coalesce(col("order_approved_at"), col("order_purchase_timestamp"))
).withColumn(
    "order_delivered_carrier_date",
    coalesce(col("order_delivered_carrier_date"), col("order_approved_at"))
)

orders_fixed.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("ecommerce_catalog.silver.orders_clean")

print("Orders timestamps fixed successfully")

from pyspark.sql.functions import col, expr

orders_df = spark.table("ecommerce_catalog.silver.orders_clean")

orders_fixed = orders_df.withColumn(
    "order_estimated_delivery_date",
    expr("coalesce(order_estimated_delivery_date, order_purchase_timestamp)")
)

orders_fixed.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("ecommerce_catalog.silver.orders_clean")


from pyspark.sql.functions import col, coalesce

orders_df = spark.table("ecommerce_catalog.silver.orders_clean")

orders_fixed = orders_df.withColumn(
    "order_delivered_customer_date",
    coalesce(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date"),
        col("order_purchase_timestamp")
    )
)

orders_fixed.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("ecommerce_catalog.silver.orders_clean")

print("Orders table fixed successfully")



from pyspark.sql.functions import col
tables = [
    "products_clean",
    "customers_clean",
    "sellers_clean",
    "orders_clean",
    "order_items_clean",
    "order_payments_clean",
    "product_category_name_translation_clean"
]
for table in tables:

    print("\n==============================")
    print(f"Checking Timestamp Columns in: {table}")
    print("==============================")
    df = spark.table(f"ecommerce_catalog.silver.{table}")
    for column, dtype in df.dtypes:
        # Check only timestamp columns
        if dtype == "timestamp":
            null_count = df.filter(col(column).isNull()).count()
            if null_count > 0:
                print(f" {column} -> {null_count} NULL timestamps found")
            else:
                print(f" {column} -> All timestamps valid")